In [ ]:
import os
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score

REPO_ROOT = Path("/content/drive/MyDrive/IMM-TSF")
PROC_DIR = REPO_ROOT / "data" / "MIMIC" / "processed"
LABEL_DIR = REPO_ROOT / "data" / "MIMIC" / "labels"


def set_seed(seed=1):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def safe_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def safe_auprc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_prob)


def build_mortality_labels(mimic_root):
    adm = pd.read_csv(Path(mimic_root) / "hosp" / "admissions.csv.gz", low_memory=False)
    adm["hadm_id"] = pd.to_numeric(adm["hadm_id"], errors="coerce")
    adm = adm.dropna(subset=["hadm_id"]).copy()
    adm["hadm_id"] = adm["hadm_id"].astype(int)

    if "hospital_expire_flag" in adm.columns:
        adm["label"] = pd.to_numeric(adm["hospital_expire_flag"], errors="coerce").fillna(0).astype(int)
    else:
        adm["deathtime"] = pd.to_datetime(adm["deathtime"], errors="coerce")
        adm["label"] = adm["deathtime"].notna().astype(int)

    out = adm[["hadm_id", "label"]].drop_duplicates().copy()
    return out


class IrregularMIMICDataset(Dataset):
    def __init__(self, processed_dir, label_df, max_len=128):
        self.processed_dir = Path(processed_dir)
        self.max_len = max_len
        self.label_map = dict(zip(label_df["hadm_id"].astype(str), label_df["label"].astype(float)))
        self.items = []

        for p in sorted(self.processed_dir.iterdir()):
            if not p.is_dir():
                continue
            rid = p.name
            ts_file = p / "time_series.csv"
            emb_files = list(p.glob("text_embeddings*.pt"))

            if rid not in self.label_map:
                continue
            if not ts_file.exists():
                continue
            if len(emb_files) == 0:
                continue

            try:
                df = pd.read_csv(ts_file)
                if len(df) < 2:
                    continue
            except Exception:
                continue

            self.items.append({
                "rid": rid,
                "ts_file": ts_file,
                "emb_file": emb_files[0],
                "label": float(self.label_map[rid]),
            })

        print("Dataset size:", len(self.items))

    def __len__(self):
        return len(self.items)

    def _get_time_col(self, df):
        candidates = ["time", "charttime", "hours_from_admit", "hours", "minute", "minutes", "timestamp"]
        for c in candidates:
            if c in df.columns:
                return c
        return None

    def __getitem__(self, idx):
        row = self.items[idx]
        df = pd.read_csv(row["ts_file"])

        time_col = self._get_time_col(df)

        non_feature_cols = {"record_id", "hadm_id", "subject_id", "storetime", "charttime", "date_time", "timestamp"}
        if time_col is not None:
            non_feature_cols.add(time_col)

        feat_cols = [c for c in df.columns if c not in non_feature_cols]
        X = df[feat_cols].apply(pd.to_numeric, errors="coerce")
        M = (~X.isna()).astype(np.float32)
        X = X.fillna(0.0).astype(np.float32)

        if time_col is not None:
            t = pd.to_numeric(df[time_col], errors="coerce").fillna(method="ffill").fillna(0).to_numpy(dtype=np.float32)
        else:
            t = np.arange(len(df), dtype=np.float32)

        # sort by time
        order = np.argsort(t)
        t = t[order]
        X = X.to_numpy(dtype=np.float32)[order]
        M = M.to_numpy(dtype=np.float32)[order]

        # delta times
        dt = np.zeros_like(t, dtype=np.float32)
        if len(t) > 1:
            dt[1:] = np.maximum(0.0, t[1:] - t[:-1])

        # truncate from the end to preserve latest irregular course
        if len(t) > self.max_len:
            t = t[-self.max_len:]
            dt = dt[-self.max_len:]
            X = X[-self.max_len:]
            M = M[-self.max_len:]

        # text embedding
        text_emb = torch.load(row["emb_file"], map_location="cpu")
        if isinstance(text_emb, dict):
            for k in ["embedding", "embeddings", "text_embedding", "hidden_states"]:
                if k in text_emb:
                    text_emb = text_emb[k]
                    break
        if torch.is_tensor(text_emb):
            text_emb = text_emb.detach().cpu().float().numpy()
        text_emb = np.asarray(text_emb, dtype=np.float32)
        if text_emb.ndim == 1:
            text_vec = text_emb
        else:
            text_vec = np.mean(text_emb, axis=0)

        y = np.float32(row["label"])

        return {
            "x": torch.tensor(X, dtype=torch.float32),
            "m": torch.tensor(M, dtype=torch.float32),
            "dt": torch.tensor(dt[:, None], dtype=torch.float32),
            "text": torch.tensor(text_vec, dtype=torch.float32),
            "y": torch.tensor([y], dtype=torch.float32),
        }


def collate_fn(batch):
    B = len(batch)
    max_len = max(item["x"].shape[0] for item in batch)
    n_feat = batch[0]["x"].shape[1]
    text_dim = batch[0]["text"].shape[0]

    x = torch.zeros(B, max_len, n_feat)
    m = torch.zeros(B, max_len, n_feat)
    dt = torch.zeros(B, max_len, 1)
    seq_mask = torch.zeros(B, max_len)
    text = torch.zeros(B, text_dim)
    y = torch.zeros(B, 1)

    for i, item in enumerate(batch):
        L = item["x"].shape[0]
        x[i, :L] = item["x"]
        m[i, :L] = item["m"]
        dt[i, :L] = item["dt"]
        seq_mask[i, :L] = 1.0
        text[i] = item["text"]
        y[i] = item["y"]

    return x, m, dt, seq_mask, text, y


class IrregularTS_Encoder(nn.Module):
    def __init__(self, n_feat, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_feat * 2 + 1, hidden_dim)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, m, dt, seq_mask):
        z = torch.cat([x, m, dt], dim=-1)
        z = self.input_proj(z)
        out, _ = self.rnn(z)

        # masked mean pooling
        seq_mask_exp = seq_mask.unsqueeze(-1)
        out = out * seq_mask_exp
        pooled = out.sum(dim=1) / seq_mask_exp.sum(dim=1).clamp_min(1.0)
        pooled = self.dropout(pooled)
        return pooled


class TextEncoder(nn.Module):
    def __init__(self, text_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, text):
        return self.net(text)


class OldGateFusionClassifier(nn.Module):
    def __init__(self, n_feat, text_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.ts_enc = IrregularTS_Encoder(n_feat, hidden_dim, dropout)
        self.txt_enc = TextEncoder(text_dim, hidden_dim, dropout)

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid(),
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, m, dt, seq_mask, text):
        h_ts = self.ts_enc(x, m, dt, seq_mask)
        h_txt = self.txt_enc(text)
        g = self.gate(torch.cat([h_ts, h_txt], dim=-1))
        h = g * h_ts + (1 - g) * h_txt
        logits = self.head(h)
        return logits


def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    ys, ps = [], []

    with torch.no_grad():
        for x, m, dt, seq_mask, text, y in loader:
            x = x.to(device)
            m = m.to(device)
            dt = dt.to(device)
            seq_mask = seq_mask.to(device)
            text = text.to(device)
            y = y.to(device)

            logits = model(x, m, dt, seq_mask, text)
            prob = torch.sigmoid(logits)

            ys.extend(y.squeeze(-1).cpu().numpy().tolist())
            ps.extend(prob.squeeze(-1).cpu().numpy().tolist())

    ys = np.array(ys)
    ps = np.array(ps)
    pred = (ps >= threshold).astype(int)

    return {
        "auc": safe_auc(ys, ps),
        "auprc": safe_auprc(ys, ps),
        "acc": accuracy_score(ys, pred),
        "f1": f1_score(ys, pred, zero_division=0),
        "n": len(ys),
        "pos_rate": float(np.mean(ys)),
        "threshold": threshold,
    }


def find_best_threshold(model, loader, device):
    model.eval()
    ys, ps = [], []

    with torch.no_grad():
        for x, m, dt, seq_mask, text, y in loader:
            x = x.to(device)
            m = m.to(device)
            dt = dt.to(device)
            seq_mask = seq_mask.to(device)
            text = text.to(device)

            logits = model(x, m, dt, seq_mask, text)
            prob = torch.sigmoid(logits)

            ys.extend(y.squeeze(-1).cpu().numpy().tolist())
            ps.extend(prob.squeeze(-1).cpu().numpy().tolist())

    ys = np.array(ys)
    ps = np.array(ps)

    best_thr, best_f1 = 0.5, -1
    for thr in np.linspace(0.05, 0.95, 19):
        pred = (ps >= thr).astype(int)
        val_f1 = f1_score(ys, pred, zero_division=0)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_thr = thr
    return best_thr


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mimic_root", type=str, required=True)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--seed", type=int, default=1)
    parser.add_argument("--max_len", type=int, default=128)
    args = parser.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    label_df = build_mortality_labels(args.mimic_root)
    LABEL_DIR.mkdir(parents=True, exist_ok=True)
    label_path = LABEL_DIR / "mortality_labels_from_irregular_runner.csv"
    label_df.to_csv(label_path, index=False)
    print("Saved labels to:", label_path)
    print(label_df["label"].value_counts(dropna=False))

    ds = IrregularMIMICDataset(PROC_DIR, label_df, max_len=args.max_len)

    labels = np.array([item["label"] for item in ds.items], dtype=int)
    idx = np.arange(len(ds))

    train_idx, temp_idx, y_train, y_temp = train_test_split(
        idx, labels, test_size=0.30, random_state=args.seed, stratify=labels
    )
    val_idx, test_idx, y_val, y_test = train_test_split(
        temp_idx, y_temp, test_size=0.50, random_state=args.seed, stratify=y_temp
    )

    train_ds = Subset(ds, train_idx)
    val_ds = Subset(ds, val_idx)
    test_ds = Subset(ds, test_idx)

    print("Train n =", len(train_ds), "pos rate =", y_train.mean())
    print("Val   n =", len(val_ds),   "pos rate =", y_val.mean())
    print("Test  n =", len(test_ds),  "pos rate =", y_test.mean())

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate_fn)

    sample = ds[0]
    n_feat = sample["x"].shape[1]
    text_dim = sample["text"].shape[0]

    model = OldGateFusionClassifier(n_feat=n_feat, text_dim=text_dim, hidden_dim=128, dropout=0.2).to(device)

    n_pos = max(1, int(y_train.sum()))
    n_neg = max(1, int(len(y_train) - n_pos))
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)
    print("Using pos_weight =", float(pos_weight.item()))

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_auc = -np.inf
    best_state = None

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0

        for x, m, dt, seq_mask, text, y in train_loader:
            x = x.to(device)
            m = m.to(device)
            dt = dt.to(device)
            seq_mask = seq_mask.to(device)
            text = text.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x, m, dt, seq_mask, text)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        val_metrics = evaluate(model, val_loader, device, threshold=0.5)
        val_auc = val_metrics["auc"]
        val_auc_cmp = -1e9 if np.isnan(val_auc) else val_auc

        if val_auc_cmp > best_val_auc:
            best_val_auc = val_auc_cmp
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={total_loss / max(1, len(train_loader)):.4f} | "
            f"val_auc={val_metrics['auc']:.4f} | "
            f"val_auprc={val_metrics['auprc']:.4f} | "
            f"val_acc={val_metrics['acc']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f}"
        )

    model.load_state_dict(best_state)
    best_thr = find_best_threshold(model, val_loader, device)
    print("Best validation threshold:", best_thr)

    test_metrics = evaluate(model, test_loader, device, threshold=best_thr)

    print("\nFinal test metrics")
    for k, v in test_metrics.items():
        print(f"{k}: {v}")

    out_dir = REPO_ROOT / "run_logs"
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out_dir / "irregular_oldgate_mortality_classifier.pt")
    print("Saved model to:", out_dir / "irregular_oldgate_mortality_classifier.pt")


if __name__ == "__main__":
    main()